In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.signal import remez, freqz
from ipywidgets import FloatSlider, Dropdown, VBox, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# PARKS–McCLELLAN FIR DESIGN FROM SPECIFICATIONS
# ============================================================

plt.ioff()

CONTENT_WIDTH = '1000px'

plt.rcParams.update({'font.size':11,'axes.titlesize':12.5,'axes.labelsize':11,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.pm-root{
    width:1000px;
    max-width:1000px;
    font-family:Arial,sans-serif;
}

.pm-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:18px;
    font-weight:bold;
}

.pm-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.55;
    margin-bottom:9px;
}

.pm-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:9px 11px;
    margin-bottom:8px;
    font-size:13.5px;
    line-height:1.48;
}

.pm-note{
    width:100%;
    box-sizing:border-box;
    background:#f7fbff;
    border-left:5px solid #1976d2;
    border-radius:5px;
    padding:10px 13px;
    margin-bottom:8px;
    font-size:13.5px;
    line-height:1.5;
}

.pm-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:6px;
}

.pm-cols{
    display:flex;
    gap:14px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.pm-col{
    flex:1;
    min-width:0;
}

.pm-table{
    width:100%;
    border-collapse:collapse;
    table-layout:fixed;
    font-size:12px;
}

.pm-table th,.pm-table td{
    border:1px solid #aac0dc;
    padding:4px 5px;
    text-align:center;
    white-space:nowrap;
}

.pm-table th{
    background:#edf3fa;
    font-weight:bold;
}

.widget-label{
    font-size:13px !important;
}

.jupyter-widgets input,
.jupyter-widgets select{
    font-size:13px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="pm-root">

<div class="pm-header">
Parks–McClellan FIR Design from Specifications
</div>

<div class="pm-doc">

<b>Purpose.</b>
This notebook illustrates the practical design of an optimum equiripple Type-I low-pass FIR filter directly from the specifications
ω<sub>p</sub>, ω<sub>s</sub>, δ<sub>p</sub> and δ<sub>s</sub>.
The Kaiser or Herrmann relation supplies only an <i>initial estimate</i> of the FIR length N.
The filter is then designed with the Parks–McClellan algorithm and its actual passband and stopband errors are measured.

<br><br>

<b>Iterative order adjustment.</b>
If the initial filter does not satisfy both specifications, N is increased by 2 and the design is repeated.
If the initial design satisfies the specifications, N is progressively decreased by 2 until the preceding design fails.
The final result is therefore the smallest odd FIR length found by this procedure that satisfies both error requirements.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

estimate_method = Dropdown(options=['Kaiser','Herrmann'],value='Kaiser',description='Estimate:',style={'description_width':'75px'},layout=Layout(width='210px'))

wp_slider = FloatSlider(value=0.40,min=0.10,max=0.75,step=0.01,description='ωp / π:',continuous_update=True,readout_format='.2f',style={'description_width':'65px'},layout=Layout(width='300px'))

ws_slider = FloatSlider(value=0.50,min=0.20,max=0.90,step=0.01,description='ωs / π:',continuous_update=True,readout_format='.2f',style={'description_width':'65px'},layout=Layout(width='300px'))

dp_slider = FloatSlider(value=0.010,min=0.001,max=0.05,step=0.001,description='δp:',continuous_update=True,readout_format='.3f',style={'description_width':'65px'},layout=Layout(width='300px'))

ds_slider = FloatSlider(value=0.005,min=0.001,max=0.05,step=0.001,description='δs:',continuous_update=True,readout_format='.3f',style={'description_width':'65px'},layout=Layout(width='300px'))

controls = VBox([
    HTML('<div class="pm-title">Design specifications</div>'),
    HBox([estimate_method,wp_slider,ws_slider]),
    HBox([dp_slider,ds_slider])
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0'))

# ============================================================
# ORDER ESTIMATION
# ============================================================

def make_odd(N):
    N = max(3,int(np.ceil(N)))
    return N if N % 2 == 1 else N+1

def kaiser_estimate(dp,ds,wp,ws):
    deltaF = (ws-wp)/2.0
    N = (-20.0*np.log10(np.sqrt(dp*ds))-13.0)/(14.6*deltaF)+1.0
    return make_odd(N)

def herrmann_estimate(dp,ds,wp,ws):
    deltaF = (ws-wp)/2.0
    dp_h,ds_h = (dp,ds) if ds < dp else (ds,dp)
    lp = np.log10(dp_h)
    ls = np.log10(ds_h)
    Dinf = (0.005309*lp**2+0.07114*lp-0.4761)*ls-(0.00266*lp**2+0.5941*lp+0.4278)
    f = 11.01217+0.51244*(lp-ls)
    N = (Dinf-f*deltaF**2)/deltaF+1.0
    return make_odd(N)

# ============================================================
# PARKS–McCLELLAN DESIGN FOR A GIVEN LENGTH
# ============================================================

def design_filter(N,wp,ws,dp,ds):
    weight_ratio = dp/ds
    h = remez(N,[0.0,wp,ws,1.0],[1.0,0.0],weight=[1.0,weight_ratio],fs=2.0,maxiter=100,grid_density=32)
    omega,H = freqz(h,worN=32768)
    fn = omega/np.pi
    mag = np.abs(H)
    pass_mask = fn <= wp
    stop_mask = fn >= ws
    dp_actual = np.max(np.abs(mag[pass_mask]-1.0))
    ds_actual = np.max(mag[stop_mask])
    passed = dp_actual <= dp and ds_actual <= ds
    return h,fn,mag,dp_actual,ds_actual,passed

# ============================================================
# ITERATIVE LENGTH SEARCH
# ============================================================

def find_required_length(method,wp,ws,dp,ds):
    N0 = kaiser_estimate(dp,ds,wp,ws) if method == 'Kaiser' else herrmann_estimate(dp,ds,wp,ws)
    trials = []

    h,fn,mag,dpa,dsa,passed = design_filter(N0,wp,ws,dp,ds)
    trials.append((N0,dpa,dsa,passed))

    if not passed:
        N = N0
        while not passed and N < 401:
            N += 2
            h,fn,mag,dpa,dsa,passed = design_filter(N,wp,ws,dp,ds)
            trials.append((N,dpa,dsa,passed))
        final_N = N

    else:
        N = N0
        final_N = N

        while N > 3:
            candidate = N-2
            h2,fn2,mag2,dpa2,dsa2,passed2 = design_filter(candidate,wp,ws,dp,ds)
            trials.append((candidate,dpa2,dsa2,passed2))

            if not passed2:
                break

            N = candidate
            final_N = candidate

    h,fn,mag,dpa,dsa,passed = design_filter(final_N,wp,ws,dp,ds)

    return N0,final_N,trials,h,fn,mag,dpa,dsa

# ============================================================
# HTML OUTPUT
# ============================================================

info_output = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

def make_trial_table(trials):
    rows = ""

    for i,(N,dpa,dsa,passed) in enumerate(trials,1):
        status = "PASS" if passed else "FAIL"
        rows += f"<tr><td>{i}</td><td>{N}</td><td>{dpa:.6f}</td><td>{dsa:.6f}</td><td><b>{status}</b></td></tr>"

    return f"""
    <div class="pm-box">
    <div class="pm-title">Iterative FIR-length search</div>
    <table class="pm-table">
    <tr><th>Trial</th><th>FIR length N</th><th>Actual δp</th><th>Actual δs</th><th>Status</th></tr>
    {rows}
    </table>
    </div>
    """

# ============================================================
# CREATE FIGURE ONLY ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(10.0,7.0))
ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# COMPLETE MAGNITUDE RESPONSE
# ============================================================

line_mag, = ax1.plot([],[],color='red',linewidth=1.4,label='Parks–McClellan response')
desired_pass, = ax1.plot([],[],'--',linewidth=1.0,label='Desired response')
desired_stop, = ax1.plot([],[],'--',linewidth=1.0)
upper_pass = ax1.axhline(1.01,linestyle=':',linewidth=1.0,label=r'$1\pm\delta_p$')
lower_pass = ax1.axhline(0.99,linestyle=':',linewidth=1.0)
stop_limit = ax1.axhline(0.005,linestyle=':',linewidth=1.0,label=r'$\delta_s$')
dontcare_patch = ax1.axvspan(0.40,0.50,alpha=0.07,label="Don't-care region")

ax1.set_xlim(0,1)
ax1.set_ylim(-0.03,1.12)
ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax1.set_ylabel(r'$|H(e^{j\omega})|$')
ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=2,frameon=False)

# ============================================================
# PASSBAND DETAIL
# ============================================================

line_pass, = ax2.plot([],[],color='red',linewidth=1.4,label='Actual response')
pass_desired = ax2.axhline(1.0,linestyle='--',linewidth=1.0,label='Desired level')
pass_upper = ax2.axhline(1.01,linestyle=':',linewidth=1.0,label=r'$1+\delta_p$')
pass_lower = ax2.axhline(0.99,linestyle=':',linewidth=1.0,label=r'$1-\delta_p$')

ax2.set_title('Passband Detail')
ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax2.set_ylabel(r'$|H(e^{j\omega})|$')
ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=2,frameon=False)

# ============================================================
# STOPBAND DETAIL
# ============================================================

line_stop, = ax3.plot([],[],color='red',linewidth=1.4,label='Actual response')
stop_spec = ax3.axhline(0.005,linestyle=':',linewidth=1.0,label=r'$\delta_s$')
stop_zero = ax3.axhline(0.0,linestyle='--',linewidth=1.0,label='Desired level')

ax3.set_title('Stopband Detail')
ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax3.set_ylabel(r'$|H(e^{j\omega})|$')
ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),ncol=2,frameon=False)

# ============================================================
# IMPULSE RESPONSE
# ============================================================

impulse_markers, = ax4.plot([],[],'ro',markersize=3.3)
stem_collection = None
center_line = ax4.axvline(0,linestyle='--',linewidth=1.0,label='Symmetry center')

ax4.set_title('Final Type-I FIR Impulse Response')
ax4.set_xlabel('Sample index $n$')
ax4.set_ylabel('$h[n]$')
ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.18),frameon=False)

plt.subplots_adjust(left=0.075,right=0.985,top=0.95,bottom=0.11,wspace=0.28,hspace=0.65)

# ============================================================
# UPDATE CALLBACK
# ============================================================

def update_design(change=None):
    global dontcare_patch,stem_collection

    method = estimate_method.value
    wp = wp_slider.value
    ws = ws_slider.value
    dp = dp_slider.value
    ds = ds_slider.value

    if ws <= wp:
        info_output.value = '<div class="pm-root"><div class="pm-box"><b>Invalid specifications:</b> ωs must be greater than ωp.</div></div>'
        return

    try:
        Nk = kaiser_estimate(dp,ds,wp,ws)
        Nh = herrmann_estimate(dp,ds,wp,ws)
        N0,Nfinal,trials,h,fn,mag,dpa,dsa = find_required_length(method,wp,ws,dp,ds)

    except Exception as e:
        info_output.value = f'<div class="pm-root"><div class="pm-box"><b>Design error:</b> {e}</div></div>'
        return

    info_output.value = f"""
    <div class="pm-root">

    <div class="pm-note">
    <b>What this notebook does.</b>
    The internal Parks–McClellan exchange iterations are carried out by
    <code>scipy.signal.remez()</code>.
    The notebook implements the outer design procedure: it estimates an initial FIR length from the specifications,
    designs the filter, measures the actual passband and stopband errors, and then adjusts the length by N±2
    until the specifications are satisfied with the smallest suitable odd length.
    </div>

    <div class="pm-box">
    <div class="pm-title">Current design</div>

    <div class="pm-cols">

    <div class="pm-col">
    Passband edge: <b>ωp = {wp:.2f}π</b><br>
    Stopband edge: <b>ωs = {ws:.2f}π</b><br>
    Transition width: <b>{ws-wp:.2f}π</b>
    </div>

    <div class="pm-col">
    Required δp: <b>{dp:.4f}</b><br>
    Required δs: <b>{ds:.4f}</b><br>
    Weight ratio Ws/Wp: <b>{dp/ds:.3f}</b>
    </div>

    <div class="pm-col">
    Kaiser estimate: <b>N = {Nk}</b><br>
    Herrmann estimate: <b>N = {Nh}</b><br>
    Selected estimate: <b>{method}</b>
    </div>

    <div class="pm-col">
    Initial length: <b>N = {N0}</b><br>
    Final length: <b>N = {Nfinal}</b><br>
    Filter order: <b>{Nfinal-1}</b>
    </div>

    </div>
    </div>

    {make_trial_table(trials)}

    <div class="pm-box">
    <div class="pm-title">Final measured errors</div>
    Actual passband error:
    <b>δp,actual = {dpa:.6f}</b>
    &nbsp;&nbsp;≤&nbsp;&nbsp;{dp:.6f}
    &nbsp;&nbsp;→&nbsp;&nbsp;
    <b>{"PASS" if dpa <= dp else "FAIL"}</b>
    <br>
    Actual stopband error:
    <b>δs,actual = {dsa:.6f}</b>
    &nbsp;&nbsp;≤&nbsp;&nbsp;{ds:.6f}
    &nbsp;&nbsp;→&nbsp;&nbsp;
    <b>{"PASS" if dsa <= ds else "FAIL"}</b>
    </div>

    </div>
    """

    # ========================================================
    # COMPLETE MAGNITUDE RESPONSE
    # ========================================================

    line_mag.set_data(fn,mag)
    desired_pass.set_data([0,wp],[1,1])
    desired_stop.set_data([ws,1],[0,0])

    upper_pass.set_ydata([1+dp,1+dp])
    lower_pass.set_ydata([1-dp,1-dp])
    stop_limit.set_ydata([ds,ds])

    dontcare_patch.remove()
    dontcare_patch = ax1.axvspan(wp,ws,alpha=0.07)

    ax1.set_ylim(-0.03,max(1.12,1+2.5*dp))
    ax1.set_title(f'Final Parks–McClellan Magnitude Response — N = {Nfinal}')

    # ========================================================
    # PASSBAND DETAIL
    # ========================================================

    pass_mask = fn <= wp

    line_pass.set_data(fn[pass_mask],mag[pass_mask])
    pass_upper.set_ydata([1+dp,1+dp])
    pass_lower.set_ydata([1-dp,1-dp])

    ax2.set_xlim(0,wp)
    ax2.set_ylim(1-max(2.0*dp,1.2*dpa),1+max(2.0*dp,1.2*dpa))

    # ========================================================
    # STOPBAND DETAIL
    # ========================================================

    stop_mask = fn >= ws

    line_stop.set_data(fn[stop_mask],mag[stop_mask])
    stop_spec.set_ydata([ds,ds])

    ax3.set_xlim(ws,1)
    ax3.set_ylim(-0.05*max(ds,dsa),1.35*max(ds,dsa))

    # ========================================================
    # IMPULSE RESPONSE
    # ========================================================

    n = np.arange(Nfinal)

    impulse_markers.set_data(n,h)

    if stem_collection is not None:
        stem_collection.remove()

    segments = [np.array([[ni,0],[ni,hi]]) for ni,hi in zip(n,h)]
    stem_collection = LineCollection(segments)
    ax4.add_collection(stem_collection)

    center = (Nfinal-1)/2

    center_line.set_xdata([center,center])
    center_line.set_label(f'Symmetry center = {int(center)}')

    ax4.set_xlim(-1,Nfinal)
    ax4.set_ylim(min(np.min(h)*1.15,-0.05),max(np.max(h)*1.15,0.05))

    fig.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

estimate_method.observe(update_design,names='value')
wp_slider.observe(update_design,names='value')
ws_slider.observe(update_design,names='value')
dp_slider.observe(update_design,names='value')
ds_slider.observe(update_design,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info_output)
display(fig.canvas)

update_design()